#### Step 1: Upload the Adapters back to Colab

- In your project folder on Google Drive (or local Colab space), create a folder called temp_adapter.

- Put your adapter_model.safetensors, adapter_config.json, and tokenizer_config.json inside it.

#### Step 2: Run the Recovery Script

Run this code. It skips training and goes straight to merging and exporting.

In [9]:
import os

# Ensure unsloth is installed by running the pip command directly
# Removing the conditional check to guarantee execution in Colab environments.
print("Ensuring unsloth is installed...")
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" --q
os.environ['UNSLOTH_INSTALL'] = 'unsloth'

from unsloth import FastLanguageModel
import torch

# 1. Configuration
# Point this to the folder where you just put your adapter files
ADAPTER_PATH = "recovery" # Reverted back to 'recovery' as files are located there.
MAX_SEQ_LENGTH = 2048

# 2. Load the Base Model + Your Adapters
# Unsloth is smart: it sees the adapter files and automatically
# loads the base Gemma model they were trained on.
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = ADAPTER_PATH,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = True,
)

# 3. Enable Fast Inference (Faster merging)
FastLanguageModel.for_inference(model)

# 4. Export to GGUF
# Save to Google Drive this time to ensure it doesn't disappear!
export_path = "/content/drive/My Drive/fine-tuning/models/final_gguf/cyber_expert_recovered" # Uncommented and defined

print(f"📦 Starting GGUF Export to: {export_path}...")
model.save_pretrained_gguf(
    "models",  # Output directory
    tokenizer,                                 # Include tokenizer
    quantization_method="q4_k_m"              # 4-bit K-quant (medium quality)
)

print("✅ Success! Your GGUF is now safe on Google Drive.")

Ensuring unsloth is installed...
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
==((====))==  Unsloth 2026.1.2: Fast Gemma2 patching. Transformers: 4.57.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


SafetensorError: Error while deserializing header: header too large